# **Laboratorijska vježba 1**: Zadaci za samostalni rad
## **Podaci iz Svjetskog izvješća o sreći**

Svjetsko izvješće o sreći godišnja je publikacija Mreže rješenja za održivi razvoj Ujedinjenih naroda (engl. *United Nations Sustainable Development Solutions Network*). Sadrži članke i ljestvice nacionalne sreće na temelju ocjena vlastitog života ispitanika, koje izvješće također povezuje s različitim životnim čimbenicima.

U ovoj laboratorijskoj vježbi istražit ćemo sreću u različitim zemljama i povezana obilježja. Skupovi podataka koje ćemo koristiti dostupni su u *Data/happiness2020.csv* i *Data/countries_info.csv*.

U nastavku je dan sažetak stupaca (značajki) u skupu podataka:

**happines2020.csv**

*   country - naziv države
*   happiness_score - ocjena sreće
*   social_support - socijalna podrška (ublažavanje učinaka nejednakosti)
*   healthy_life_expectancy - očekivanji zdravi životni vijek
*   freedom_of_choices - sloboda donošenja životnih izbora
*   generosity - velikodušnost (milosrđe, volonteri)
*   perceptrion_of_corruption - percepcija korupcije
*   world_region - regija države u svijetu

**countries_info.csv**

*   country_name - naziv države
*   area - površina u kvadratnim miljama
*   population - broj stanovnika
*   literacy - postotak pismenih stanovnika




In [ ]:
!head Data/countries_info.csv

country_name,area,population,literacy
afghanistan,647500,31056997,"36,0"
albania,28748,3581655,"86,5"
algeria,2381740,32930091,"70,0"
argentina,2766890,39921833,"97,1"
armenia,29800,2976372,"98,6"
australia,7686850,20264082,"100,0"
austria,83870,8192880,"98,0"
azerbaijan,86600,7961619,"97,0"
bahrain,665,698585,"89,1"


In [ ]:
import pandas as pd

DATA_FOLDER = 'Data/'

HAPPINESS_DATASET = DATA_FOLDER+"happiness2020.csv"
COUNTRIES_DATASET = DATA_FOLDER+"countries_info.csv"

## **Zadatak 1: Učitavanje podataka**

Učitajte 2 skupa podataka u Pandas DataFrame-ove (zvane *happiness* i *countries*) te prikažite prve retke. Koristite ispravnu metodu za učitavanje podataka i provjerite jesu li podaci u očekivanom formatu.

In [ ]:
# Mjesto za Vaš kod
happiness = pd.read_csv(HAPPINESS_DATASET)
countries = pd.read_csv(COUNTRIES_DATASET)

happiness.head(), countries.head()

## **Zadatak 2: Spajanje podataka**

Kreirajte DataFrame pod nazivom *country_features* spajanjem prethodno stvorena 2 DataFrame-a. Redak novog DataFrame-a mora opisivati sve značajke koje imamo o nekoj državi. Pri spajanju pripazite jesu li nazivi stupaca i formati podataka u oba skupa podataka kompatibilni.


In [ ]:
# Mjesto za Vaš kod
country_features = pd.merge(happiness, countries, on="country")

country_features.head()

## **Zadatak 3: Gdje su ljudi sretniji?**

Ispišite prvih 10 država na temelju njihove ocjene sreće (što je veća ocjena, država je sretnija).


In [ ]:
# Mjesto za Vaš kod
country_features.sort_values("score", ascending=False).head(10)

Zanima nas u kojoj su regiji svijeta ljudi sretniji.

Izradite i ispišite DataFrame s:
1.   Brojem zemalja za svaku svjetsku regiju.
2.   Prosječnom ocjenom sreće


Sortirajte rezultat da biste prikazali rangiranje sreće.

In [ ]:
# Mjesto za Vaš kod
region_stats = country_features.groupby("region").agg(
    country_count=("country", "count"),
    avg_happiness=("score", "mean")
)

region_stats.sort_values("avg_happiness", ascending=False)

Najbolje rangirana regija ima samo nekoliko država! Koje su to i koji je njihov rezultat sreće?

In [ ]:
# Mjesto za Vaš kod
best_region = region_stats.sort_values("avg_happiness", ascending=False).index[0]

country_features[country_features["region"] == best_region][["country", "score"]]

## **Zadatak 4: Koliko je svijet pismen?**

Ispišite države koje su u prvih 10% najpismenijih država svijeta.

Za svaku zemlju ispišite naziv i regiju svijeta u formatu: {ime regije}-{ime zemlje} ({ocjena sreće}).


In [ ]:
# Mjesto za Vaš kod
threshold = country_features["literacy"].quantile(0.9)

top_literacy = country_features[country_features["literacy"] >= threshold]

for _, row in top_literacy.iterrows():
    print(f"{row['region']}-{row['country']} ({row['literacy']})")

Koliki je globalni prosjek razine pismenosti?

In [ ]:
# Mjesto za Vaš kod
country_features["literacy"].mean()

Izračunajte udio zemalja s razinom pismenosti iznad 50%. Ispišite vrijednost u postotcima, formatiranu s 2 decimale.

In [ ]:
# Mjesto za Vaš kod
share = (country_features["literacy"] > 50).sum() / len(country_features) * 100
print(f"{share:.2f}%")

Ispišite ukupan broj stanovnika na svijetu koji su nepismeni (pismenost im je manja od 50%) te postotak nepismenog svjetskog stanovništva.

In [ ]:
# Mjesto za Vaš kod
illiterate = country_features[country_features["literacy"] < 50]

illiterate_population = illiterate["population"].sum()
world_population = country_features["population"].sum()

percentage = (illiterate_population / world_population) * 100

print("Nepismeni stanovnici:", illiterate_population)
print(f"Udio nepismenih: {percentage:.2f}%")

## **Zadatak 5: Gustoća naseljenosti**

Dodajte DataFrame-u *country_features* novi stupac zvan *population_density* koji se dobije dijeljenjem stupca *population* sa stupcem *area*.

In [ ]:
# Mjesto za Vaš kod
country_features["population_density"] = (
    country_features["population"] / country_features["area"]
)

Koji je rezultat sreće za države koje su u zadnjih 10% po gustoći naseljenosti?

In [ ]:
# Mjesto za Vaš kod
density_threshold = country_features["population_density"].quantile(0.9)

country_features[country_features["population_density"] >= density_threshold][["country", "score"]]

## **Zadatak 6: Zdravi i sretni?**

Iscrtajte na raspršeni dijagram (engl. *scatter plot*) ocjenu sreće (x os) i očekivano trajanje zdravog života (y os). Komentirajte odnos između ove dvije varijable. Napomena: za crtanje dijagrama moguće je koristiti metodu *plot* DataFrame objetka unutar biblioteke Pandas.

In [ ]:
# Mjesto za Vaš kod
import matplotlib.pyplot as plt

plt.scatter(country_features["score"], country_features["healthy_life_expectancy"])

plt.xlabel("Happiness score")
plt.ylabel("Healthy life expectancy")
plt.title("Happiness vs Healthy Life Expectancy")

plt.show()